In [ ]:
import pickle

with open('../../data/generated/bm_200p_10000t_rp.pkl', 'rb') as file:
    data = pickle.load(file)

data.head()

,coords,num_steps,trajectory_type,attributes
0,"[[0.0, -8.728432979232437, -11.308386906074094...",200,Brownian Motion,"{'generator_type': 'numpy', 'delta_t': 4.61636..."
1,"[[0.0, -0.34801025991661605, -1.10295344069430...",200,Brownian Motion,"{'generator_type': 'numpy', 'delta_t': 3.22171..."
2,"[[0.0, 1.1021095406564758, 5.872228498024922, ...",200,Brownian Motion,"{'generator_type': 'numpy', 'delta_t': 4.73036..."
3,"[[0.0, 2.4113575699800203, -1.7202773175335349...",200,Brownian Motion,"{'generator_type': 'numpy', 'delta_t': 2.40849..."
4,"[[0.0, 0.048623725796051315, 0.035220166950249...",200,Brownian Motion,"{'generator_type': 'numpy', 'delta_t': 0.20256..."


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# 6. Definir la arquitectura CNN
class TrajectoryCNN(nn.Module):
    def __init__(self, input_channels, seq_length, num_classes):
        super().__init__()
        self.first_conv = nn.Conv1d(input_channels, 32, kernel_size=3, padding=1)
        self.conv1_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)
        self.conv2_32 = nn.Conv1d(32, 32, kernel_size=3, padding=1)
        self.conv_32_64 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.conv1_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv2_64 = nn.Conv1d(64, 64, kernel_size=3, padding=1)
        self.conv_64_128 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.conv1_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.conv2_128 = nn.Conv1d(128, 128, kernel_size=3, padding=1)
        self.max_pool = nn.MaxPool1d(2)
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 4))
        self.fc1 = nn.Linear(4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    
    def forward(self, data):
        loop = 2
        a_resnet = self.first_conv(data)
        a = self.conv1_32(a_resnet)
        a = self.conv2_32(a)
        a = a + a_resnet
        a = self.max_pool(a)
        b_resnet = self.conv_32_64(a)
        b = self.conv1_64(b_resnet)
        b = self.conv2_64(b)
        b = b + b_resnet
        b = self.max_pool(b)
        c_resnet = self.conv_64_128(b)
        c = self.conv1_128(c_resnet)
        c = self.conv2_128(c)
        c = c + c_resnet
        c = self.avg_pool(c)
        c = c.view(c.size(0), -1)  # Aplanar la salida
        c = F.relu(self.fc1(c))
        c = self.fc2(c)
        return c